# End-to-End with Falcon AI: Trace → Debug → Evaluate → Dataset → Fix in One Workflow

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/falcon-ai-page/falcon-ai/end-to-end.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on_GitHub-181717?logo=github&logoColor=white)](https://github.com/future-agi/cookbooks/blob/cookbook/falcon-ai-page/falcon-ai/end-to-end.ipynb)

| Time | Difficulty |
|------|------------|
| 15 min | Beginner |

In one Falcon AI chat: find failing traces, capture them as a regression dataset, score the baseline, get a paste-ready prompt fix, and watch the eval scores recover after applying it. You walk away with a fixed agent and a reusable regression suite that catches the same failure pattern next time.

**Prerequisites:**
- FutureAGI account: [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see [Get your API keys](https://docs.futureagi.com/docs/admin-settings))
- A traced project with mixed-quality traces. If you don't have one, instrument any agent with the `Add tracing` step below.
- Python 3.10+
- OpenAI API key (`OPENAI_API_KEY`)


## Install


In [ ]:
%pip install fi-instrumentation-otel traceai-openai openai

In [ ]:
import os
os.environ["FI_API_KEY"] = "your-fi-api-key"
os.environ["FI_SECRET_KEY"] = "your-fi-secret-key"
os.environ["OPENAI_API_KEY"] = "your-openai-key"

## Step 1: Add tracing to your agent

Three lines send every LLM call and tool invocation to FutureAGI as structured spans. `OpenAIInstrumentor` auto-instruments the OpenAI SDK; wrap your agent's entry point with `@tracer.agent` so each request becomes one parent span.


In [ ]:
from fi_instrumentation import register, FITracer
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor

trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="falcon-ai-end-to-end",
)
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)
tracer = FITracer(trace_provider.get_tracer("falcon-ai-end-to-end"))

In [ ]:
from openai import OpenAI

client = OpenAI()


# Replace this with your own agent's entry point.
# The @tracer.agent decorator makes each call show up as one parent span
# in your FutureAGI Tracing project, with the OpenAI calls nested underneath.
@tracer.agent(name="my_agent")
def my_agent(user_message: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a customer support assistant for an electronics store. Answer questions about products and orders."},
            {"role": "user", "content": user_message},
        ],
    )
    return response.choices[0].message.content


# A support agent without any grounding tool is likely to fabricate specifics
# (tracking numbers, return windows, warranty lengths) when asked about them.
# This gives Falcon AI a failing trace to analyze in the next step.
print(my_agent("Where is order ORD-12345?"))
print(my_agent("What\'s your return policy for opened wireless headphones?"))

trace_provider.force_flush()

Open **Tracing** → your project. Once traces are flowing, move to the next step. For broader instrumentation patterns see [Manual Tracing](https://docs.futureagi.com/docs/cookbook/quickstart/manual-tracing).

## Step 2: /analyze-trace-errors on the project

Stay on the Tracing page so Falcon AI picks up the project as context, then open the sidebar and type:

> Analyze trace errors in this project

> **Tip.** `Cmd+K` (Mac) or `Ctrl+K` (Windows) opens Falcon AI from anywhere in the dashboard, with the current page auto-attached as a context chip.

Falcon AI runs `analyze_project_traces` across the project, classifies issues against an error taxonomy (Hallucination, Wrong Intent, Tool Misuse, etc.), and scores each trace 1 to 5.

![Falcon AI sidebar showing the analyze trace errors completion card with per-trace scores and the dominant error category](https://fi-cookbook-assets.s3.ap-south-1.amazonaws.com/falcon-ai/end-to-end/step-4-analyze-trace-errors.png)

Switch to the **Feed** tab in Tracing to see the same findings rendered per-trace, with the exact quote that triggered each finding.



## Step 3: /build-dataset to capture the failing rows

Same conversation. Lock the bad traces into a regression set so any future fix is evaluated against the same failures.

> Build me a dataset called `falcon-demo-failures` with the queries from the traces flagged with Hallucinated Content. Columns: `query` (text), `agent_output` (text), `failure_category` (text).

A completion card appears with a link to the new dataset.

![Falcon AI completion card showing the new falcon-demo-failures dataset with row count and link](https://fi-cookbook-assets.s3.ap-south-1.amazonaws.com/falcon-ai/end-to-end/step-5-build-dataset.png)

Open **Datasets** → `falcon-demo-failures` to confirm the rows.


## Step 4: /run-evaluations to get a baseline

Same conversation. Score the dataset so you have a number to beat after the fix.

> Run `factual_accuracy` and `completeness` evals on the `falcon-demo-failures` dataset.

![Falcon AI evaluation results card showing per-row scores for factual_accuracy and completeness](https://fi-cookbook-assets.s3.ap-south-1.amazonaws.com/falcon-ai/end-to-end/step-6-run-evaluations.png)

Expect `factual_accuracy` to be in the floor and `completeness` to be high: the agent fully addresses each question, but the answers are invented.


## Step 5: /fix-with-falcon to get the prompt diff

Open one of the worst-scoring traces from the Feed. With that trace as context, type:

> /fix-with-falcon

Falcon AI reads the actual system prompt and model output from the span and returns one concrete change in a fixed format: *Current* then *Replace with* under 400 words.

![Falcon AI fix-with-falcon output with sections for What happened, Root cause in the agent, The fix (current vs replace with), and Expected score improvement](https://fi-cookbook-assets.s3.ap-south-1.amazonaws.com/falcon-ai/end-to-end/step-7-fix-with-falcon.png)

If the OpenAI auto-instrumentor didn't capture the literal system message, the **Current** block is flagged as inferred. The fix is still load-bearing because the failure mode (ungrounded specifics) is independent of the exact wording.


## Step 6: Apply the fix and verify scores recover

Paste the **Replace with** block as your new system prompt and re-run the same queries through your traced agent. Then back in Falcon AI:

> Re-run the same evals on `falcon-demo-failures` and compare to the previous run.

Sample after-fix scores (your numbers will vary):

| Eval | Before | After |
|---|---|---|
| **factual_accuracy** | 1 / 5 | 5 / 5 |
| **completeness** | 5 / 5 | 5 / 5 |

`factual_accuracy` recovers because the agent no longer fabricates. `completeness` stays high because the refusal still addresses the user's question.


> **Check.** Trace → Debug → Evaluate → Dataset → Fix, all driven from one chat. Every artifact (dataset, eval run, prompt diff) is saved as a clickable completion card.

## Explore further

- **[Context-Aware Trace Debugging](/docs/cookbook/falcon-ai/context-aware-debugging)**: From a single bad trace to a paste-ready prompt fix in minutes
- **[Building Evaluation Datasets from Production Traces](/docs/cookbook/falcon-ai/eval-datasets-from-traces)**: Curate balanced eval sets from real traces with `/build-dataset`
- **[Falcon AI Skills](/docs/falcon-ai/features/skills)**: All built-in slash commands and how to write your own